# 02 — GR-input robustness: controlled corruption of the conditioning signal

**No training.** The gain-prior model's defining property is that its conditioning
input is a *physical quantity* (dB gain), so we can perturb it in physically
meaningful ways and measure the output degradation — something impossible with a
learned latent interface. Every perturbation runs through **two paths**:

1. **`model`** — the trained gain-prior network (Δg + coloration active);
2. **`bypass`** — plain `amplitude_match(dry, gr_perturbed)` (the multiply alone).

If the model path degrades *more slowly* than the bypass path, the network is
actively absorbing GR error (the Δg head generalises beyond the label's RMS smear);
if the two curves are parallel, the network passes upstream error straight through.

Perturbation families (magnitude sweeps):

| family | what it models |
|---|---|
| `noise` (frame-rate Gaussian, σ in dB) | stochastic predictor error — validates the §5.3 first-order claim (ε dB → ≈ 11.5 %·ε amplitude) |
| `bias` (constant dB offset) | calibration error between GR conventions / devices |
| `scale` (gr × α) | compression-depth error (wrong ratio estimate upstream) |
| `lag` (time shift, ms) | attack/release timing error — perceptually the critical axis |
| `smooth` (moving average, ms) | a slower upstream detector than the label's 23 ms window |
| `quant` (step, dB) | a coarsely quantised GR transport (e.g. 0.5 dB metering) |

> Segments (60 s of 2 validation pairs) keep the sweep tractable; ≈ 5–10 min on CPU.

In [1]:
# -- 0. Setup ------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from exp_common import (
    METRIC_COLS, SR, aggregate_rows, amplitude_match, ensure_eval_out,
    load_gain_prior, load_pair, load_split, pairs_from_keys, params_for,
    score_signal, stream_gain_prior,
)

SELECTED_GAIN_PRIOR_RUN = "gain_prior_20260702_085618_diffssl_lstm32_gain_prior"
MODEL, HP, RUN_DIR = load_gain_prior(SELECTED_GAIN_PRIOR_RUN)
SPLIT = load_split(RUN_DIR)
VAL_PAIRS = pairs_from_keys(SPLIT.val_pair_keys)
OUT = ensure_eval_out()

PAIRS = [VAL_PAIRS[2], VAL_PAIRS[7]]     # two settings, one val song
START_SEC, DUR_SEC = 30.0, 60.0
HOP = 256                                 # frame rate for band-limited GR noise

SEGMENTS = []
for song, setting in PAIRS:
    dry, wet, gr = load_pair(setting, song, start_sec=START_SEC, duration_sec=DUR_SEC)
    SEGMENTS.append({"song": song, "setting": setting, "dry": dry, "wet": wet,
                     "gr": gr, "params": params_for(setting)})
    print(f"{song} / {setting}: {dry.shape[-1]/SR:.0f} s")

/Volumes/Saola's Drive/AllCode/thesis/Virtual-Analogue-Compressor-Modelling/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 8,322-param GainPriorDiffSSLLSTM  (gain_prior_20260702_085618_diffssl_lstm32_gain_prior / best-059-730800.ckpt)
Ecstasy / threshold_-4_attack_10_release_0.1_ratio_2: 60 s
Ecstasy / threshold_4_attack_10_release_0.1_ratio_10: 60 s


In [2]:
# -- 1. Perturbation families -------------------------------------------
# Each entry: family -> (magnitudes, fn(gr [1,T], mag) -> gr' [1,T]).
# GR noise is generated at FRAME rate (172 Hz) and linearly upsampled - white
# per-sample noise would be unphysically fast for a gain trajectory and is
# mostly wiped by the model's own input clamp.

def _noise(gr, sigma_db, seed=0):
    T = gr.shape[-1]
    g = torch.Generator().manual_seed(seed)
    n_frames = T // HOP + 2
    n = torch.randn(1, 1, n_frames, generator=g) * sigma_db
    n = F.interpolate(n, size=T, mode="linear", align_corners=False)
    return gr + n.squeeze(0)

def _bias(gr, db):
    return gr + db

def _scale(gr, alpha):
    return gr * alpha

def _lag(gr, ms):
    s = int(round(abs(ms) / 1000 * SR))
    if s == 0:
        return gr
    if ms > 0:   # GR arrives LATE
        return torch.cat([gr[..., :1].expand(1, s), gr[..., :-s]], dim=-1)
    return torch.cat([gr[..., s:], gr[..., -1:].expand(1, s)], dim=-1)

def _smooth(gr, ms):
    W = max(3, int(round(ms / 1000 * SR)) | 1)          # odd width
    k = torch.ones(1, 1, W) / W
    return F.conv1d(gr.unsqueeze(0), k, padding=W // 2).squeeze(0)[..., :gr.shape[-1]]

def _quant(gr, step_db):
    return torch.round(gr / step_db) * step_db

FAMILIES = {
    "noise (sigma dB)":  ([0.1, 0.25, 0.5, 1.0, 2.0, 4.0], _noise),
    "bias (dB)":         ([-3.0, -1.0, -0.5, 0.5, 1.0, 3.0], _bias),
    "scale (x)":         ([0.5, 0.75, 0.9, 1.1, 1.25, 1.5], _scale),
    "lag (ms)":          ([-100.0, -23.0, -5.0, 5.0, 23.0, 100.0], _lag),
    "smooth (ms)":       ([5.0, 23.0, 58.0, 116.0, 232.0], _smooth),
    "quant (dB step)":   ([0.5, 1.0, 2.0, 3.0], _quant),
}

In [ ]:
# -- 2. Sweep: every (family x magnitude) through both paths -------------
rows = []

def eval_variant(gr_fn, mag, family):
    for path in ("model", "bypass"):
        chunk_rows = []
        for seg in SEGMENTS:
            gr_p = gr_fn(seg["gr"], mag) if gr_fn else seg["gr"]
            n = min(seg["dry"].shape[-1], gr_p.shape[-1])
            if path == "bypass":
                pred = amplitude_match(seg["dry"][..., :n], gr_p[..., :n])
            else:
                pred = stream_gain_prior(MODEL, seg["dry"][..., :n], gr_p[..., :n],
                                         seg["params"])
            chunk_rows += score_signal(seg["dry"], pred, seg["wet"][..., :n])
        rows.append({"Family": family, "Magnitude": mag, "Path": path,
                     **aggregate_rows(chunk_rows)})
        print(f"{family:18s} mag={mag:8.2f} {path:6s} "
              f"GR MAE {rows[-1]['GR MAE (dB)']:.3f} dB  "
              f"MR-STFT {rows[-1]['MR-STFT']:.3f}")

eval_variant(None, 0.0, "clean")                       # baselines
for family, (mags, fn) in FAMILIES.items():
    for mag in mags:
        eval_variant(fn, mag, family)

df = pd.DataFrame(rows)
df.to_csv(OUT / "02_gr_robustness.csv", index=False)
print(f"\nSaved -> {OUT}/02_gr_robustness.csv")
df[df.Family == "clean"].round(4)

clean              mag=    0.00 model  GR MAE 0.022 dB  MR-STFT 0.029
clean              mag=    0.00 bypass GR MAE 0.054 dB  MR-STFT 0.074
noise (sigma dB)   mag=    0.10 model  GR MAE 0.054 dB  MR-STFT 0.043
noise (sigma dB)   mag=    0.10 bypass GR MAE 0.072 dB  MR-STFT 0.078
noise (sigma dB)   mag=    0.25 model  GR MAE 0.124 dB  MR-STFT 0.077
noise (sigma dB)   mag=    0.25 bypass GR MAE 0.119 dB  MR-STFT 0.092
noise (sigma dB)   mag=    0.50 model  GR MAE 0.243 dB  MR-STFT 0.133
noise (sigma dB)   mag=    0.50 bypass GR MAE 0.210 dB  MR-STFT 0.125
noise (sigma dB)   mag=    1.00 model  GR MAE 0.465 dB  MR-STFT 0.234
noise (sigma dB)   mag=    1.00 bypass GR MAE 0.404 dB  MR-STFT 0.200
noise (sigma dB)   mag=    2.00 model  GR MAE 0.893 dB  MR-STFT 0.418
noise (sigma dB)   mag=    2.00 bypass GR MAE 0.813 dB  MR-STFT 0.363
noise (sigma dB)   mag=    4.00 model  GR MAE 1.803 dB  MR-STFT 0.783
noise (sigma dB)   mag=    4.00 bypass GR MAE 1.708 dB  MR-STFT 0.713
bias (dB)          m

In [ ]:
# -- 3. Degradation curves per family -------------------------------------
# Solid = model, dashed = bypass; horizontal lines = clean baselines. Where the
# solid curve sits below the dashed one, the network is absorbing GR error.

PLOT_METRICS = ["GR MAE (dB)", "MR-STFT", "M_NRMSE"]
clean = df[df.Family == "clean"].set_index("Path")

fig, axes = plt.subplots(len(PLOT_METRICS), len(FAMILIES),
                         figsize=(3.1 * len(FAMILIES), 3.0 * len(PLOT_METRICS)),
                         squeeze=False)
for j, family in enumerate(FAMILIES):
    sub = df[df.Family == family]
    for i, m in enumerate(PLOT_METRICS):
        ax = axes[i, j]
        for path, ls, color in (("model", "-", "#d62728"), ("bypass", "--", "#1f77b4")):
            s = sub[sub.Path == path].sort_values("Magnitude")
            ax.plot(s["Magnitude"], s[m], ls, marker="o", ms=3, color=color, label=path)
            ax.axhline(clean.loc[path, m], color=color, lw=0.6, alpha=0.5)
        ax.grid(alpha=0.3)
        if i == 0:
            ax.set_title(family, fontsize=9)
        if j == 0:
            ax.set_ylabel(m, fontsize=9)
        if i == len(PLOT_METRICS) - 1:
            ax.set_xlabel("magnitude", fontsize=8)
axes[0, 0].legend(fontsize=8)
fig.suptitle("GR-input corruption sensitivity - model (solid) vs raw multiply (dashed)")
fig.tight_layout()
fig.savefig(OUT / "02_gr_robustness_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# -- 4. First-order check: eps dB -> 11.5%*eps amplitude error ------------
# MODEL_GAIN_PRIOR.md section 5.3 predicts d(gain)/d(dB) = ln10/20 = 0.115 per dB
# BEFORE any correction. The bypass path's normalised envelope error (M_NRMSE)
# under bias eps should track 0.115*|eps|; the model path's gap below it is the
# absorbed fraction.

bias = df[df.Family == "bias (dB)"].copy()
bias["abs_eps"] = bias["Magnitude"].abs()
eps = np.linspace(0, bias["abs_eps"].max() * 1.05, 50)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(eps, np.abs(10 ** (eps / 20) - 1), "k--", lw=1,
        label=r"first order: $|10^{\epsilon/20}-1| \approx 0.115\,\epsilon$")
for path, color in (("bypass", "#1f77b4"), ("model", "#d62728")):
    s = bias[bias.Path == path]
    ax.scatter(s["abs_eps"], s["M_NRMSE"], color=color, label=f"{path} (M_NRMSE)")
ax.set_xlabel(r"GR bias $|\epsilon|$ (dB)"); ax.set_ylabel("normalised envelope error")
ax.grid(alpha=0.3); ax.legend()
ax.set_title("Does GR error enter the output at the predicted first-order rate?")
fig.tight_layout()
fig.savefig(OUT / "02_first_order_check.png", dpi=150, bbox_inches="tight")
plt.show()

# absorbed fraction per family (GR MAE): 1 - (model degradation / bypass degradation)
print("Absorbed fraction of GR-induced degradation (GR MAE), per family:")
for family in FAMILIES:
    sub = df[df.Family == family]
    dm = (sub[sub.Path == "model"]["GR MAE (dB)"].mean()
          - clean.loc["model", "GR MAE (dB)"])
    db_ = (sub[sub.Path == "bypass"]["GR MAE (dB)"].mean()
           - clean.loc["bypass", "GR MAE (dB)"])
    frac = 1 - dm / db_ if db_ > 1e-9 else float("nan")
    print(f"  {family:18s} bypass +{db_:.3f} dB | model +{dm:.3f} dB | absorbed {frac:5.1%}")

## Reading the results

- **`noise`** is the closest proxy for real predictor error: the 05 detector's test
  MAE is 0.268 dB, so its column of this sweep (σ ≈ 0.25–0.5) predicts the cascade
  degradation measured in notebook 01 — cross-check the two.
- **`lag`** isolates timing: compression that is depth-correct but late is the
  perceptually damaging failure mode (pumping). If a +23 ms lag (one label window)
  costs more than 1 dB of GR MAE, the Δ-timing loss term upstream is well motivated.
- **`smooth`** connects to `oracle_w4096` in notebook 01: both hand the model a
  slower prior. If the model path recovers most of the loss, Δg genuinely re-sharpens
  ballistics rather than memorising the 23 ms convention.
- The **absorbed fraction** table is the single number to quote: how much of an
  upstream conditioning error the gain-application stage corrects, per error type.